# Look at our data
Reads the saved splits in `data/` and shows what's in them. It only reads, never changes anything.
No GPU and no downloads needed.

In [ ]:
import os, sys

if os.path.exists("/content"):   # on Colab: get the latest code + data from GitHub
    if not os.path.exists("/content/mfr-dpo"):
        !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
    !git -C /content/mfr-dpo pull -q
    REPO = "/content/mfr-dpo"
else:                            # on your laptop, running from notebooks/
    REPO = ".."

sys.path.insert(0, f"{REPO}/src")
import importlib, mfr_data
importlib.reload(mfr_data)

import pandas as pd
splits = mfr_data.load_splits(f"{REPO}/data")   # splits["helpful"]["train"] is a table
DATASETS = ["helpful", "safe", "quality"]

## How many pairs?

In [ ]:
pd.DataFrame({name: {split: len(df) for split, df in splits[name].items()} for name in DATASETS})

## Sanity checks
Every check should say OK. If one fails, something went wrong when the splits were made.

In [ ]:
all_rows = pd.concat([df.assign(dataset=name, split=split)
                      for name in DATASETS for split, df in splits[name].items()])

checks = {
    "every id is unique": all_rows["id"].is_unique,
    "every prompt appears only once (no leaks between splits or datasets)": all_rows["prompt"].is_unique,
    "no empty prompt / chosen / rejected": (all_rows[["prompt", "chosen", "rejected"]].apply(lambda c: c.str.strip() != "")).all().all(),
    "chosen and rejected are never the same": (all_rows["chosen"] != all_rows["rejected"]).all(),
    "every pair fits in 1024 tokens": ((all_rows["prompt_tokens"] + all_rows[["chosen_tokens", "rejected_tokens"]].max(axis=1)) <= 1024).all(),
}
for name, ok in checks.items():
    print("OK  " if ok else "FAIL", name)

## How long are the pairs? (in tokens)
Length of the prompt plus the longer of the two responses, for the train splits. The dashed line is our 1024-token limit.

In [ ]:
import matplotlib.pyplot as plt

train = all_rows[all_rows["split"] == "train"].copy()
train["pair_tokens"] = train["prompt_tokens"] + train[["chosen_tokens", "rejected_tokens"]].max(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharex=True, sharey=True)
for ax, name in zip(axes, DATASETS):
    lengths = train.loc[train["dataset"] == name, "pair_tokens"]
    ax.hist(lengths, bins=range(0, 1025, 32), color="#2a78d6", edgecolor="white", linewidth=1)
    ax.axvline(1024, color="#6b6a66", linestyle="--", linewidth=1)
    ax.set_title(f"{name}  (median {int(lengths.median())} tokens)", fontsize=11, loc="left")
    ax.set_xlabel("tokens per pair")
    ax.grid(axis="y", color="#e6e5e0", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
axes[0].set_ylabel("pairs")
plt.tight_layout()
plt.show()

In [ ]:
# the same thing as numbers: prompt, chosen and rejected lengths (train splits)
train.groupby("dataset")[["prompt_tokens", "chosen_tokens", "rejected_tokens"]].describe(percentiles=[0.5, 0.95]) \
     .loc[DATASETS].T.round().astype(int)

## Is the chosen answer just longer?
DPO can learn a shortcut: "longer answer = better answer". If the chosen response is longer in most pairs,
we should keep this in mind when reading results (and when we look at the model's answers).

In [ ]:
pd.DataFrame({
    name: {
        "chosen is longer": f'{100 * (df["chosen_tokens"] > df["rejected_tokens"]).mean():.0f}% of pairs',
        "median chosen tokens": int(df["chosen_tokens"].median()),
        "median rejected tokens": int(df["rejected_tokens"].median()),
    }
    for name in DATASETS for df in [splits[name]["train"]]
})

## Read some examples
Run this cell again to see new random examples. You can also pick one: `show("safe", "val", i=5)`.

In [ ]:
import random, textwrap

def show(dataset, split="train", i=None, width=110, max_chars=1200):
    df = splits[dataset][split]
    i = random.randrange(len(df)) if i is None else i
    row = df.iloc[i]
    def wrap(text):
        text = text[:max_chars] + (" [...]" if len(text) > max_chars else "")
        return "\n".join(textwrap.fill(line, width) for line in text.splitlines())
    print(f"==================== {row['id']} ====================")
    for part in ("prompt", "chosen", "rejected"):
        print(f"\n--- {part.upper()} ({row[part + '_tokens']} tokens) ---")
        print(wrap(row[part]))
    print()

for name in DATASETS:
    show(name)

## Search the prompts

In [ ]:
def search(word, dataset, split="train"):
    df = splits[dataset][split]
    return df.loc[df["prompt"].str.contains(word, case=False), ["id", "prompt"]]

search("python", "helpful")